In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime
# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # 컬럼 전체 보기
pd.set_option('display.width', 0) 

#데이터 불러오기 
df = pd.read_csv('model_df.csv')
print(df.columns)
df.head(1)

Index(['기획년도', '주차', '카테고리', '라인', '시즌이월', '시즌', '복종', '소품종', '성별', '총입고수량',
       '판매수량', '판매액', '평균택가', '평균원가', '총입고원가', '총입고택가', '매출원가계', '판매택가계',
       '주차별_평균_실판매가', '월', '월별_평균_실판매가', '시즌별_평균_실판매가', '실판매가', '할인율',
       '누적판매수량', '누적판매액', '누적매출원가', '누적판매택가', '누적판매율', 'ROI', '맑음', '흐림', '비',
       '강한 비', '눈', '강한 눈', '진눈깨비', '악천후일수', '평균기온(도)'],
      dtype='object')


,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도)
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693,986743.0,1.0,53,52297400,10838500,52947000,3.31,0.11,4,0,0,0,3,0,0,0,-9.1


In [3]:
# 주차 기준 정렬 먼저
df = df.sort_values(['카테고리', '주차'])

# 할인율이 음수인 경우, 같은 카테고리 내에서 ffill
df['할인율'] = df.groupby('카테고리')['할인율'].transform(
    lambda x: x.mask(x < 0).ffill()
)
# df[df['할인율']<0]

In [ ]:

# 피벗테이블로 4년동안 데이터가 있는 카테고리만 남김
# 카테고리별 연도 존재 여부 확인 (0: 없음, 1: 있음)
category_year_table = df.groupby(["카테고리", "기획년도"]).size().unstack(fill_value=0)
# 연도가 존재하면 1로 변환 (카테고리가 존재했음을 의미)
category_year_table = (category_year_table > 0).astype(int)

# 21, 22, 23, 24년도 모두 존재한 카테고리만 필터링
categories_all_years = category_year_table[
    (category_year_table.get(2021, 0) == 1) & 
    (category_year_table.get(2022, 0) == 1) & 
    (category_year_table.get(2023, 0) == 1) & 
    (category_year_table.get(2024, 0) == 1)
].index

# 새로운 데이터프레임 생성
df = df[df["카테고리"].isin(categories_all_years)].copy()
df['카테고리'].nunique()

# 2023년도 데이터(지난 1년동안의 데이터)로 카테고리 선정하고자 함
# 판매 중간에 연도가 바뀌면서 잘린 겨울 제품 제외함함
df = df[df['기획년도'] == 2023]
# df[(df['기획년도'] == 2023) & (df['시즌'] != '겨울')]
# df['카테고리'].nunique()
df

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도),연도주차,가격_변화율,판매량_변화율,가격탄력성(판매량변화율/가격변화율)
961,2021,2021-07-11,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,10931,0,0,69900,8312,90858472,764076900,0,0,NaN,7,64118.0,31895,64118.0,8.0,0,0,0,0,0.00,0.00,4,3,0,0,0,0,0,0,28.5,2021-28,NaN,NaN,NaN
1001,2021,2021-07-18,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,10931,56,3131520,69900,8312,90858472,764076900,465472,3914400,55920.0,7,64118.0,31895,55920.0,20.0,56,3131520,465472,3914400,0.51,0.03,5,1,0,1,0,0,0,1,29.8,2021-29,NaN,inf,NaN
1028,2021,2021-07-25,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,10931,-7,-506220,69900,8312,90858472,764076900,-58184,-489300,72317.0,7,64118.0,31895,72317.0,20.0,49,2625300,407288,3425100,0.45,0.02,5,1,1,0,0,0,0,0,30.6,2021-30,29.322246,-112.500000,-3.836677
1064,2021,2021-08-01,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,35541,-26,-1530810,69900,7420,263731990,2484315900,-192933,-1817400,58877.0,8,40120.0,31895,58877.0,16.0,23,1094490,214355,1607700,0.06,0.00,4,1,2,0,0,0,0,0,28.0,2021-31,-18.584842,271.428571,-14.604836
1101,2021,2021-08-08,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,42541,425,23612190,69900,7465,317568565,2973615900,3172625,29707500,55529.0,8,40120.0,31895,55558.0,21.0,448,24706680,3386980,31315200,1.05,0.06,4,2,1,0,0,0,0,0,27.1,2021-32,-5.686431,-1734.615385,305.044654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7928,2024,2024-09-01,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,84786,1464,44311219,99000,11930,1011539373,8393814000,17466252,144936000,31256.0,9,28631.0,43278,30267.0,69.0,65507,2796847685,776775233,6485193000,77.26,1.75,3,2,2,0,0,0,0,0,26.2,2024-35,-15.464921,-9.517923,0.615452
7982,2024,2024-09-08,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,84786,895,26426499,99000,11930,1011539373,8393814000,10677798,88605000,29519.0,9,28631.0,43278,29527.0,70.0,66402,2823274184,787453031,6573798000,78.32,1.76,3,2,2,0,0,0,0,0,27.6,2024-36,-5.557333,-38.866120,6.993664
8054,2024,2024-09-15,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,84786,631,18149165,99000,11930,1011539373,8393814000,7528146,62469000,28771.0,9,28631.0,43278,28763.0,71.0,67033,2841423349,794981177,6636267000,79.06,1.77,3,1,2,1,0,0,0,1,26.5,2024-37,-2.533961,-29.497207,11.640749
8127,2024,2024-09-22,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,84786,500,14325172,99000,11930,1011539373,8393814000,5965250,49500000,28570.0,9,28631.0,43278,28650.0,71.0,67533,2855748521,800946427,6685767000,79.65,1.77,5,1,1,0,0,0,0,0,22.2,2024-38,-0.698620,-20.760697,29.716718


In [5]:
# 기준 1) ----------------------------------------------------------------------------------------------------------------
# 매출 기여도가 높은 핵심 카테고리 (판매량 & 매출 기준)
top_sales = df.groupby("카테고리", group_keys=False).agg(
    총판매수량=("판매수량", "sum"),
    총판매액=('판매액', 'sum'),
    총매출원가=('매출원가계', 'sum'),    
    총입고수량=("총입고수량", "max")
).reset_index()
top_sales['총순수익'] = top_sales['총판매액'] - top_sales['총매출원가']

# 기준 2) ----------------------------------------------------------------------------------------------------------------
# 재고 부담 & ROI 개선이 필요한 카테고리 (최대 누적판매율)
top_inventory_ROI = df.groupby("카테고리", group_keys=False).agg(
    최대누적판매율=("누적판매율", "max"),  
    평균할인율=("할인율", "mean"),
    ROI=("ROI", 'max')
).reset_index()

# 기준 3) ---------------------------------------------------------------------------------------------------------
# 할인 전략 개선이 필요한 카테고리(평균 할인율 변화 기준)
# 시즌 종료 시 할인율 상승 패턴 확인 (평균 비교 방식)
df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 기준 3) ---------------------------------------------------------------------------------------------------------
# 할인 전략 개선이 필요한 카테고리 (평균 할인율 변화 기준)
# 시즌 종료 시 할인율 상승 패턴 확인 (평균 비교 방식)

df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 마지막 한달간 할인율 증가량
def calc_discount_diff(group):
    weekly_discount = group.groupby("연도주차", as_index=False)["할인율"].mean()
    last_4 = weekly_discount.tail(4)
    before = weekly_discount.iloc[:-4]
    last_mean = last_4['할인율'].mean()
    before_mean = before['할인율'].mean() if not before.empty else 0
    return pd.Series({
        "마지막4주_평균할인율": last_mean,
        "이전_평균할인율": before_mean,
        "시즌종료_할인율증가량": last_mean - before_mean
    })
discount_rise = df.groupby("카테고리", group_keys=False).apply(calc_discount_diff).reset_index()

# 마지막 주차 할인율
last_week_discount = df.loc[df.groupby("카테고리")["주차"].idxmax()][["카테고리", "할인율"]]
last_week_discount = last_week_discount.rename(columns={"할인율": "마지막주차_할인율"})

# 기준 4) ----------------------------------------------------------------------------------------------------------------
# 가격탄력성이 높은 카테고리 (할인율-판매량 상관관계)
top_correlation = df.groupby("카테고리", group_keys=False).apply(
    lambda x: x["할인율"].corr(x["판매수량"])
).reset_index()
top_correlation.columns = ["카테고리", "가격탄력성(할인율-판매량_상관관계)"]

# 가격_변화율과 판매량_변화율을 이용한 가격 탄력성 계산
df["가격_변화율"] = df.groupby('카테고리')['주차별_평균_실판매가'].pct_change() * 100
df["판매량_변화율"] = df.groupby('카테고리')['판매수량'].pct_change() * 100

df["가격탄력성(판매량변화율/가격변화율)"] = df["판매량_변화율"] / df["가격_변화율"]
df["가격탄력성(판매량변화율/가격변화율)"] = df["가격탄력성(판매량변화율/가격변화율)"].replace([np.inf, -np.inf], np.nan)

elastic_cate = df.groupby('카테고리')['가격탄력성(판매량변화율/가격변화율)'].mean().reset_index()

# 모든 데이터 병합 ----------------------------------------------------------------------------------------------------------------
category_selection = (
    top_sales
    .merge(top_inventory_ROI, on="카테고리", how="left")
    .merge(last_week_discount, on="카테고리", how="left")
    .merge(top_correlation, on="카테고리", how="left")
    .merge(discount_rise, on="카테고리", how="left")
    .merge(elastic_cate, on="카테고리", how="left")
    .fillna(0)  
)

# 소수점 라운딩
category_selection["평균할인율"] = category_selection["평균할인율"].round(2)
category_selection["가격탄력성(할인율-판매량_상관관계)"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].round(2)
category_selection["가격탄력성(판매량변화율/가격변화율)"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].round(2)
category_selection["시즌종료_할인율증가량"] = category_selection["시즌종료_할인율증가량"].astype(int)
category_selection["마지막주차_할인율"] = category_selection["마지막주차_할인율"].astype(int)

# 정렬 및 순위 계산 (최대 누적판매율은 낮을수록 우선순위 → 오름차순, 나머지는 높을수록 우선순위 → 내림차순)
category_selection["총매출_순위"] = category_selection["총판매액"].rank(method="min", ascending=False).astype(int)
category_selection["총순수익_순위"] = category_selection["총순수익"].rank(method="min", ascending=False).astype(int)
category_selection["총입고_순위"] = category_selection["총입고수량"].rank(method="min", ascending=False).astype(int)
category_selection["최대누적판매율_순위"] = category_selection["최대누적판매율"].rank(method="min", ascending=True).astype(int)  # 낮을수록 재고 부담 ↑
category_selection["ROI_순위"] = category_selection["ROI"].rank(method="min", ascending=True).astype(int)  # 낮을수록 수익 손해 ↑
category_selection["평균할인율_순위"] = category_selection["평균할인율"].rank(method="min", ascending=False).astype(int)
category_selection["시즌종료_할인율증가량_순위"] = category_selection["시즌종료_할인율증가량"].rank(method="min", ascending=False).astype(int)
category_selection["가격탄력성(상관계수) 순위"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].rank(method="min", ascending=False).astype(int) 
category_selection["가격탄력성(판매량변화율) 순위"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].rank(method="min", ascending=True).astype(int) # 낮을수록 탄력성 좋음음

# 각 기준별 점수 계산 ----------------------------------------------------------------------------------------------------------------
# 기준 1: 매출 기여도 높은 카테고리 (총매출, 총입고수량)
category_selection["기준1_점수"] = category_selection["총매출_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["총입고_순위"].rank(method="min", ascending=True).astype(int)
                                
# 기준 2: 재고 부담 & ROI 낮은 카테고리 (누적판매율, ROI)
category_selection["기준2_점수"] = category_selection["최대누적판매율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["ROI_순위"].rank(method="min", ascending=True).astype(int)

# 기준 3: 할인 전략 개선이 필요한 카테고리 (최대누적판매율, 평균할인율, 시즌종료_할인율증가량)
category_selection["기준3_점수"] = category_selection["평균할인율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["시즌종료_할인율증가량_순위"].rank(method="min", ascending=True).astype(int)
     
# 기준 4: 가격탄력성이 높아 할인율 최적화 효율이 좋은 카테고리 (상관계수와 판매량변화율로 본 가격탄력성)
category_selection["기준4_점수"] = category_selection["가격탄력성(상관계수) 순위"].rank(method="min", ascending=True).astype(int) + \
                                 category_selection["가격탄력성(판매량변화율) 순위"].rank(method="min", ascending=True).astype(int)

# 기준별 순위 계산 (오름차순, 낮을수록 우선순위)
category_selection["기준1_순위"] = category_selection["기준1_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준2_순위"] = category_selection["기준2_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준3_순위"] = category_selection["기준3_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준4_순위"] = category_selection["기준4_점수"].rank(method="min", ascending=True).astype(int)

# 최종 순위 계산 (기준 1, 2, 3, 4 순위를 모두 합산)
category_selection["최종_순위"] = category_selection["기준1_순위"] + category_selection["기준2_순위"] + category_selection["기준3_순위"] + category_selection["기준4_순위"]
category_selection["최종_순위"] = category_selection["최종_순위"].rank(method="min", ascending=True).astype(int)

# 최종 출력: 순위만 포함 ----------------------------------------------------------------------------------------------------------------
final_rank_output = category_selection[[
    "카테고리","기준1_순위", "기준2_순위", "기준3_순위", "기준4_순위", "최종_순위","총판매수량","총판매액","총순수익","총입고수량","최대누적판매율","ROI","평균할인율","마지막주차_할인율",
    "시즌종료_할인율증가량","가격탄력성(할인율-판매량_상관관계)","가격탄력성(판매량변화율/가격변화율)","총매출_순위","총순수익_순위","총입고_순위","최대누적판매율_순위","ROI_순위",
    "평균할인율_순위","시즌종료_할인율증가량_순위","가격탄력성(상관계수) 순위","가격탄력성(판매량변화율) 순위"]].sort_values(by="최종_순위")

display(final_rank_output) #최종순위대로 정렬
# 기준 점수를 기준으로 정렬, 하나씩 주석 해제하면서 정렬
# final_rank_output.sort_values(by=["기준1_순위"])
# final_rank_output.sort_values(by=["기준2_순위"])
# final_rank_output.sort_values(by=["기준3_순위"])
# final_rank_output.sort_values(by=["기준4_순위"])

,카테고리,기준1_순위,기준2_순위,기준3_순위,기준4_순위,최종_순위,총판매수량,총판매액,총순수익,총입고수량,최대누적판매율,ROI,평균할인율,마지막주차_할인율,시즌종료_할인율증가량,가격탄력성(할인율-판매량_상관관계),가격탄력성(판매량변화율/가격변화율),총매출_순위,총순수익_순위,총입고_순위,최대누적판매율_순위,ROI_순위,평균할인율_순위,시즌종료_할인율증가량_순위,가격탄력성(상관계수) 순위,가격탄력성(판매량변화율) 순위
120,여름_우븐 셔츠_캐쥬얼셔츠_ZB,4,76,16,11,1,241253,6266358722,4471233319,220857,75.89,1.75,61.00,76,15,0.45,-30.84,12,13,2,91,72,10,61,36,20
103,여름_니트 셔츠_라운드_ZB,1,86,23,10,2,778665,13364910973,8189398649,739995,78.16,1.79,64.15,77,12,0.47,-28.63,2,5,1,98,75,4,79,33,22
26,겨울_사파리_패딩사파리_ZB,5,37,22,72,3,78974,10475987058,6139075562,55852,63.09,0.65,57.82,73,16,0.69,209.11,6,8,14,65,18,31,50,10,132
118,여름_우븐 셔츠_드레스셔츠_ZB,11,74,52,5,4,179331,4591495332,3513740246,106473,66.74,1.93,58.12,69,11,0.56,-101.43,22,20,6,73,84,27,91,23,7
23,겨울_니트 셔츠_후드티_ZB,61,17,38,28,5,12493,424696816,276940492,22000,38.83,0.75,58.83,70,12,0.63,-1.52,83,84,39,22,22,21,79,13,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,사계절_우븐 셔츠_드레스셔츠_ZC,133,23,133,123,129,390,11087895,8149245,1000,39.00,0.95,28.67,27,-1,0.03,18.50,133,133,133,23,31,128,128,107,116
113,여름_스웨터_T-에리_ZA,65,121,99,129,130,18911,901381303,670790881,10633,87.47,2.41,55.36,55,3,-0.02,28.50,62,63,68,122,110,46,119,114,121
99,사계절_자켓_싱글재킷_ZB,77,112,131,95,131,9286,1103537284,843998123,5145,78.50,2.51,42.86,43,0,0.31,13.32,55,54,103,100,119,92,126,62,110
94,사계절_수트_블레이져(수트)_ZC,50,124,118,125,132,15817,2506992568,1799705145,11107,90.44,2.43,37.42,49,11,0.09,131.52,36,36,67,128,112,106,91,100,127


In [6]:
final_rank_output.to_csv("category_rank(소품종,겨울포함).csv", index=False, encoding="utf-8-sig")